# wm_probe.ipynb

Analyse des représentations apprises par LeWorldModel.

Ce notebook tourne **uniquement en local**. Il nécessite :
- un checkpoint `.pt` dans `local_runs/checkpoints/`
- le dataset `.npz` dans `local_runs/datasets/`

Analyses :
1. PCA 2D des embeddings (colorée par position agent / cible)
2. Linear probes : prédire position agent et cible depuis z
3. Qualité de prédiction du prédicteur
4. Sensibilité aux actions

## 1. Config

In [ ]:
from pathlib import Path

# ── À adapter selon ton run ─────────────────────────────────────────────────
CHECKPOINT_PATH = Path("./local_runs/checkpoints/lewm_vit_50k_v2.pt")
DATASET_PATH    = Path("./local_runs/datasets/dataset_iso_N5_50k.npz")
# ────────────────────────────────────────────────────────────────────────────

print(f"Checkpoint : {CHECKPOINT_PATH}  →  existe : {CHECKPOINT_PATH.exists()}")
print(f"Dataset    : {DATASET_PATH}  →  existe : {DATASET_PATH.exists()}")

## 2. Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from src.model import LeWorldModel

print("Imports OK")

## 3. Chargement du modèle et du dataset

In [ ]:
# Modèle
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
cfg  = ckpt["config"]

model = LeWorldModel(
    img_size        = cfg["img_size"],
    patch_size      = cfg["patch_size"],
    embed_dim       = cfg["embed_dim"],
    encoder_depth   = cfg["encoder_depth"],
    predictor_depth = cfg["predictor_depth"],
    n_heads         = cfg["n_heads"],
    n_actions       = cfg["n_actions"],
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Modèle chargé — embed_dim={cfg['embed_dim']}")

# Dataset
d = np.load(DATASET_PATH)
obs_t    = d["obs_t"]     # (N, 128, 128) uint8
obs_t1   = d["obs_t1"]
actions  = d["actions"]   # (N,) int64
agent_t  = d["agent_t"]   # (N, 2) int8  — (col, row)
agent_t1 = d["agent_t1"]
target   = d["target"]    # (N, 2) int8

N = len(actions)
print(f"Dataset : {N:,} transitions")

## 4. Encodage de toutes les observations

On encode `obs_t` pour obtenir les embeddings z ∈ R^D.

In [ ]:
BATCH = 512

def encode_all(obs_array):
    """obs_array : (N, H, W) uint8  →  (N, D) float32"""
    N = len(obs_array)
    embs = []
    with torch.no_grad():
        for i in range(0, N, BATCH):
            batch = torch.from_numpy(obs_array[i:i+BATCH]).float() / 255.0
            # (B, 1, H, W) → encode → (B, D)
            z = model.encoder(batch.unsqueeze(1))
            embs.append(z.cpu().numpy())
    return np.concatenate(embs, axis=0)

Z = encode_all(obs_t)   # (N, D)
print(f"Embeddings : {Z.shape}  std_mean={Z.std(axis=0).mean():.3f}  mean_abs={np.abs(Z.mean(axis=0)).mean():.4f}")

## 5. PCA 2D

On projette les 50k embeddings en 2D et on les colore par propriété de l'état.

**À observer :** est-ce que les états proches dans le gridworld sont proches dans l'espace latent ?

In [ ]:
pca = PCA(n_components=2)
Z2  = pca.fit_transform(Z)   # (N, 2)
print(f"Variance expliquée : PC1={pca.explained_variance_ratio_[0]:.1%}  PC2={pca.explained_variance_ratio_[1]:.1%}")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

props = [
    (agent_t[:, 0],  "col agent",  "Blues"),
    (agent_t[:, 1],  "row agent",  "Reds"),
    (target[:, 0],   "col cible",  "Greens"),
    (target[:, 1],   "row cible",  "Oranges"),
]

for ax, (values, title, cmap) in zip(axes.flatten(), props):
    sc = ax.scatter(Z2[:, 0], Z2[:, 1], c=values, cmap=cmap, s=1, alpha=0.3)
    plt.colorbar(sc, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

plt.suptitle("PCA des embeddings LeWM", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Linear probes

On entraîne une régression Ridge **linéaire** sur les embeddings gelés pour prédire :
- la position de l'agent (col, row)
- la position de la cible (col, row)

**Interprétation :** un R² élevé signifie que l'information est encodée **linéairement** dans l'espace latent — c'est le signe d'une bonne représentation structurée.

In [ ]:
Z_tr, Z_te, agt_tr, agt_te, tgt_tr, tgt_te = train_test_split(
    Z, agent_t.astype(float), target.astype(float),
    test_size=0.2, random_state=42
)

results = {}
for name, y_tr, y_te in [
    ("agent_col",  agt_tr[:, 0], agt_te[:, 0]),
    ("agent_row",  agt_tr[:, 1], agt_te[:, 1]),
    ("target_col", tgt_tr[:, 0], tgt_te[:, 0]),
    ("target_row", tgt_tr[:, 1], tgt_te[:, 1]),
]:
    reg = Ridge(alpha=1.0)
    reg.fit(Z_tr, y_tr)
    pred = reg.predict(Z_te)
    r2  = r2_score(y_te, pred)
    mae = np.abs(pred - y_te).mean()
    results[name] = {"R2": r2, "MAE": mae}
    print(f"{name:12s} | R²={r2:.3f}  MAE={mae:.3f} cases")

print()
print("R² proche de 1.0 = information encodée linéairement → bonne représentation")

## 7. Qualité de prédiction du prédicteur

Pour chaque transition, on encode obs_t → z_t, on prédit ẑ_{t+1}, et on encode obs_t1 → z_{t+1}.
On mesure la distance L2 entre ẑ_{t+1} et z_{t+1}.

On segmente par cas : **déplacement** vs **collision** (agent n'a pas bougé).

In [ ]:
Z_t1 = encode_all(obs_t1)   # (N, D) — embeddings réels de t+1

BATCH = 512
pred_errors = []

with torch.no_grad():
    for i in range(0, N, BATCH):
        z_b   = torch.from_numpy(Z[i:i+BATCH])          # (B, D)
        act_b = torch.from_numpy(actions[i:i+BATCH].astype(np.int64))
        # séquence T=2 : (B, 2, D)
        emb_seq = z_b.unsqueeze(1).expand(-1, 2, -1).clone()
        pred    = model.predictor(emb_seq, act_b)        # (B, 2, D)
        z_pred  = pred[:, 0].cpu().numpy()               # prédiction à t+1
        z_real  = Z_t1[i:i+BATCH]
        err     = np.linalg.norm(z_pred - z_real, axis=1)
        pred_errors.append(err)

pred_errors = np.concatenate(pred_errors)

moved    = ~np.all(agent_t == agent_t1, axis=1)
at_goal  = np.all(agent_t == target, axis=1)

print(f"Erreur L2 moyenne         : {pred_errors.mean():.4f}")
print(f"Erreur — déplacements     : {pred_errors[moved].mean():.4f}")
print(f"Erreur — collisions bord  : {pred_errors[~moved].mean():.4f}")
print(f"Erreur — agent sur cible  : {pred_errors[at_goal].mean():.4f}")

## 8. Sensibilité aux actions

Pour un état fixe, on calcule les 4 prédictions (une par action).
Si le prédicteur a appris quelque chose, les 4 prédictions doivent être différentes.

In [ ]:
ACTION_NAMES = {0: "haut", 1: "bas", 2: "gauche", 3: "droite"}

# Prendre un état central (agent au centre du board)
center_mask = (agent_t[:, 0] == 2) & (agent_t[:, 1] == 2)
if center_mask.sum() == 0:
    idx = 0
else:
    idx = np.where(center_mask)[0][0]

z_ref = torch.from_numpy(Z[idx]).unsqueeze(0)   # (1, D)
print(f"État de référence : agent={tuple(agent_t[idx])}  cible={tuple(target[idx])}")

preds = {}
with torch.no_grad():
    for a in range(4):
        act   = torch.tensor([a])
        seq   = z_ref.unsqueeze(1).expand(-1, 2, -1).clone()
        pred  = model.predictor(seq, act)
        preds[a] = pred[0, 0].numpy()   # (D,)

# Distance entre chaque paire de prédictions
print("\nDistances L2 entre prédictions :")
for a1 in range(4):
    for a2 in range(a1+1, 4):
        d = np.linalg.norm(preds[a1] - preds[a2])
        print(f"  {ACTION_NAMES[a1]:6s} vs {ACTION_NAMES[a2]:6s} : {d:.4f}")

# PCA des 4 prédictions + état réel
all_vecs = np.stack([Z[idx]] + [preds[a] for a in range(4)])
pca2 = PCA(n_components=2).fit(Z)
pts  = pca2.transform(all_vecs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(Z2[:, 0], Z2[:, 1], c="lightgray", s=1, alpha=0.2)
ax.scatter(pts[0, 0], pts[0, 1], c="black",  s=150, zorder=5, label="z_t (état réel)")
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]
for a in range(4):
    ax.scatter(pts[a+1, 0], pts[a+1, 1], c=colors[a], s=100, zorder=5,
               label=f"pred {ACTION_NAMES[a]}")
ax.legend(fontsize=9)
ax.set_title("Prédictions par action dans l'espace PCA")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()